In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import stats
import math
import glob
import pandas as pd

In [ ]:
#Sensor display approach - detection with the "Full Model"

import math
from scipy import stats

folder = "images_MP"
files = glob.glob(folder+"/*.png")
files.sort()

coef_S = 1.3
coef_H = 1


df = pd.DataFrame() 
idx = 0

curve_fit = [-6.10326225e-03,  1.66362622e-01, -1.72945339e+00,  8.20177410e+00, -1.15768530e+01, -5.80801635e+01,  2.44648954e+02]

def CubicEq(EV):
    """Funció polinòmica directa."""
    return np.polyval(curve_fit, EV)

# Precalcula valors d’EV amb una taula
EV_range = np.linspace(0, 8, 1000)
V_lookup = CubicEq(EV_range)/2.55
V_to_EV = np.interp  

#print (EV_range)
#print (V_lookup)



for f in files:
    
    index = 1
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print (f)
        
    size = img.shape
    img_hsv = np.zeros((size[0], size[1], 3))
        
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            maxi = max(RGB)
            mini = min(RGB)
            
            if maxi == mini:
                mini = maxi-1
            
            if maxi == RGB[0]:
                H = 60*((G-B)/(maxi-mini))
            elif maxi == RGB[1]:
                H = 120+60*((B-R)/(maxi-mini))
            elif maxi == RGB[2]:
                H = 240+60*(R-G)/(maxi-mini)
                
            if maxi == 0:
                S = 0
            else:
                #s = 100*((maxi-mini)/maxi) 
                S = 100*(float((maxi-mini)/maxi))
            
            V = (maxi/255)*100  
            
            img_hsv[i,j,2] = V
            img_hsv[i,j,1] = S
            img_hsv[i,j,0] = H

    if "287" in f:
        colors = ([302, 344], [626, 305], [493, 254], [507, 253], [627, 265])

    elif "307" in f:
        colors = ([160, 653], [252, 817], [220, 712], [219, 584], [138, 659], [302, 852], [165, 710],[242, 570])

    elif "305" in f:
        colors = ([466, 276], [400, 313], [519, 326], [1013, 521], [420, 367])  

    elif "291" in f:
        colors = ([526, 708], [531, 394], [420, 607], [380, 475], [450, 707])
        
    for c in colors:
                
        x = c[0]
        y = c[1]
        
        winsz = 8   #ajustar en f(x) del soroll/textura

        color_roi = (255,0,0)
        
        #img_roi = []
        
        img = cv2.imread(f)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)

        half = winsz // 2
        roi = img_hsv[x-half:x+half, y-half:y+half, :]
        h_roi, s_roi, v_roi = [ch.flatten() for ch in cv2.split(roi)]

        # Elimina outliers (vectoritzat)
        def filter_outliers(a, k=3.16):
            m, s = np.mean(a), np.std(a)
            mask = (a > m - k*s) & (a < m + k*s)
            return a[mask]
        
        h_no_out = filter_outliers(h_roi)
        s_no_out = filter_outliers(s_roi)
        v_no_out = filter_outliers(v_roi)

        h, s, v = np.mean(h_no_out), np.mean(s_no_out), np.mean(v_no_out)
        h_noise, s_noise, v_noise = np.ptp(h_no_out), np.ptp(s_no_out), np.ptp(v_no_out)
        
        rang_H = -0.08*s + 6.88
        rang_S = 0.12*s + 0.56

        c1 = (h-(abs(rang_H/2)+(h_noise/2))*coef_H, s-(rang_S/3)-(s_noise/2), 0)
        c2 = (h+(abs(rang_H/2)+(h_noise/2))*coef_H, s+((rang_S)+(s_noise/2))*coef_S, 100)
        
        # Màscara de detecció vectoritzada
        mask = cv2.inRange(img_hsv, c1, c2)
        result = cv2.bitwise_and(img_hsv, img_hsv, mask=mask)
        #result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        
        # Fase d’ajust EV → S (també vectoritzada)
        v_vals = result[:,:,2].flatten()
        
        EVv = np.interp(v_vals, V_lookup[::-1], EV_range[::-1])
        EV0 = np.interp(v, V_lookup[::-1], EV_range[::-1])
        EVvmin = np.interp((v_vals-v_noise/2), V_lookup[::-1], EV_range[::-1])
        EVvmax = np.interp((v_vals+v_noise/2), V_lookup[::-1], EV_range[::-1])
        EV = np.abs(EVv - EV0)
        EVmin = np.abs(EVvmin-EV0)
        EVmax = np.abs(EVvmax-EV0)
        
        print("EV: ", EV)
        print("EVmin: ", EVmin)
        print("EVmax: ", EVmax)
        
        s_ev = ((rang_S/2) * EV + s)
        s_ev_min = ((rang_S/2) * EVmin + s)
        s_ev_max = ((rang_S/2) * EVmax + s)
        var_s_ev = s_ev_min - s_ev_max
        
        print("s_ev: ", s_ev)
        print("s_ev_min: ", s_ev_min)
        print("s_ev_max: ", s_ev_max)
        
        print("s_noise: ", s_noise)
        print("sv_noise: ", var_s_ev)
        
        s_min = s_ev - ((s_noise)/2+(var_s_ev)/2)*coef_S
        s_max = s_ev + ((s_noise)/2+(var_s_ev)/2)*coef_S
        S_flat = result[:,:,1].flatten()
        valid = (S_flat >= s_min) & (S_flat <= s_max)
 
        
        final_hsv = result.copy().reshape(-1,3)
        #final_hsv[~valid] = 0
        final_hsv = final_hsv.reshape(result.shape)
        
        mask_opt = cv2.inRange(final_hsv[:,:,2], 1, 100)
        #mask_opt = cv2.cvtColor(mask_opt, cv2.COLOR_HSV2GRAY)
        
        final_rgb = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_opt)
        
        #mask_opt = cv2.bitwise_and(result, final_hsv)
        #final_rgb = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_opt)
        

        df.at[idx, "Image"] = f
        df.at[idx, "color"] = str(index)
        df.at[idx, "H"] = h
        df.at[idx, "S"] = s
        df.at[idx, "V"] = v
        df.at[idx, "H noise"] = h_noise
        df.at[idx, "S noise"] = s_noise
        df.at[idx, "coef H"] = coef_H
        df.at[idx, "H range eq"] = rang_H
        df.at[idx, "H range apl"] = (rang_H+h_noise)*coef_H
        df.at[idx, "Coef S"] = coef_S
        df.at[idx, "S range eq"] = rang_S
        df.at[idx, "S range apl"] = (rang_S+s_noise/2)*coef_S+rang_S/3+s_noise/2

        idx += 1
        
                    
        fig = plt.figure(figsize = (40,18))
        plt.subplot(1, 3, 1)
        plt.imshow(img_roi)
        plt.subplot(1,3,2)
        plt.imshow(result)
        plt.subplot(1,3,3)
        plt.imshow(final_rgb)
        plt.show()
        
         
        final_bgr = cv2.cvtColor(final_rgb, cv2.COLOR_RGB2BGR)
        final_name = "../results/"+f[-13:-10]+"_optimized_"+str(index)+".png"
        cv2.imwrite (final_name, final_bgr)

        index += 1

print(df)
df.to_excel("../results/Method_results.xlsx") 

In [ ]:
#Sensor-display approach -- detection "only noise"

from scipy import stats

folder = "images_MP"
files = glob.glob(folder+"/*.png")
files.sort()

coef_S = 1.3
coef_H = 1


df = pd.DataFrame() 
idx = 0

for f in files:
    
    index = 1
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print (f)
        
    size = img.shape
    img_hsv = np.zeros((size[0], size[1], 3))
        
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            maxi = max(RGB)
            mini = min(RGB)
            
            if maxi == mini:
                mini = maxi-1
            
            if maxi == RGB[0]:
                H = 60*((G-B)/(maxi-mini))
            elif maxi == RGB[1]:
                H = 120+60*((B-R)/(maxi-mini))
            elif maxi == RGB[2]:
                H = 240+60*((R-G)/(maxi-mini)) 
                
            if maxi == 0:
                S = 0
            else:
                #s = 100*((maxi-mini)/maxi) 
                S = 100*(float((maxi-mini)/maxi))
            
            V = (maxi/255)*100  
            
            img_hsv[i,j,2] = V
            img_hsv[i,j,1] = S
            img_hsv[i,j,0] = H

    if "287" in f:
        colors = ([302, 344], [626, 305], [493, 254], [507, 253], [627, 265])

    elif "307" in f:
        colors = ([160, 653], [252, 817], [220, 712], [219, 584], [138, 659], [302, 852], [165, 710],[242, 570])

    elif "305" in f:
        colors = ([466, 276], [400, 313], [519, 326], [1013, 521], [420, 367])  

    elif "291" in f:
        colors = ([526, 708], [531, 394], [420, 607], [380, 475], [450, 707])
    
    for c in colors:
                
        x = c[0]
        y = c[1]
        
        winsz = 8   #ajustar en f(x) del soroll/textura

        color_roi = (255,0,0)
        
        #img_roi = []
        
        img = cv2.imread(f)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)

        h_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        #h_circ_np = (h_roi_np)*2
        
        h_roi = h_roi_np.flatten()
        h_mean = np.mean(h_roi)
        h_std = np.std(h_roi)
        #print (h_roi)

        h_min = np.min(h_roi)
        h_max = np.max(h_roi)
        print (h_min,   h_max)

        if h_min<0 and h_max>200:
            for i in h_roi:
                if i<0:
                    index = h_roi.index(i)
                    print (index)
                    j = 360 + i
                    print (j)
                    print (h_roi[index])
                    h_roi[index] = int(j)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((h_roi > (h_mean+k*h_std)) | (h_roi < (h_mean-k*h_std)))[0]
        print ("H outliers:",  outlier_indices)
        h_no_out = np.delete(h_roi, outlier_indices)
        
        h = np.mean(h_no_out)
        h_min = np.min(h_no_out)
        h_max = np.max(h_no_out)
        h_noise = h_max-h_min
        #std_h = stats.circstats.circstd(h_circ, method='angular')
        print ("H: ", h)
        

        s_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
       #Descartar outliers amb mètode Chevyshev
        s_roi = s_roi_np.flatten()
        s_mean = np.mean(s_roi)
        s_std = np.std(s_roi)
        #print (h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((s_roi > (s_mean+k*s_std)) | (s_roi < (s_mean-k*s_std)))[0]
        print ("S outliers:",  outlier_indices)
        s_no_out = np.delete(s_roi, outlier_indices)

        s = np.mean(s_no_out)
        std_s = np.std(s_no_out)
        min_s = np.min(s_no_out)
        max_s = np.max(s_no_out)
        s_noise = max_s-min_s 
        print ("S: ", s)


        v_roi = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        v = np.mean(v_roi)
        v_max = np.max(v_roi)
        std_v = np.std(v_roi)
        print ("V: ", v)
        

        rang_H = -0.08*s + 6.88  #Són les equacions de la 90D
        rang_S = 0.12*s + 0.56 
        
        print ("Rang H eq: ", rang_H)
        print ("H noise: ", h_noise)
        print ("Rang S eq: ", rang_S)
        print ("S noise: ", s_noise)

    
        c1 = (h-h_noise/2, s-s_noise/2, 0) #per sota només resto 1/3 part del rang_S (perquè no estic mesurant a la zona més lluminosa) i el noise perquè rang S és assimètric
        c2 = (h+h_noise/2, s+s_noise/2, v_max)
    
        print(c1,c2)

        mask = cv2.inRange(img_hsv, c1, c2)
        result = cv2.bitwise_and(img, img, mask=mask)
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        result_name = "../results/"+f[-13:-10]+"_noise_"+str(winsz)+"_"+str(index)+".png"
        cv2.imwrite (result_name, result)


        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_RGB2BGR)
        img_roi_name = "../results/"+f[-13:-10]+"ROI"+str(index)+".png"
        cv2.imwrite (img_roi_name, img_roi_rgb)

        df.at[idx, "Image"] = f
        df.at[idx, "color"] = str(index)
        df.at[idx, "H"] = h
        df.at[idx, "S"] = s
        df.at[idx, "V"] = v
        df.at[idx, "H noise"] = h_noise
        df.at[idx, "S noise"] = s_noise
        df.at[idx, "coef H"] = coef_H
        df.at[idx, "H range eq"] = rang_H
        df.at[idx, "H range apl"] = (rang_H+h_noise)*coef_H
        df.at[idx, "Coef S"] = coef_S
        df.at[idx, "S range eq"] = rang_S
        df.at[idx, "S range apl"] = (rang_S+s_noise/2)*coef_S+rang_S/3+s_noise/2

        idx += 1
        
        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(img_roi)
        plt.subplot(1,2,2)
        plt.imshow(result_bgr)
        plt.show()
        
    
        index += 1

print(df)
df.to_excel("../results/noise_results.xlsx") 

In [ ]:
#Only noise (\eta)segmentation in albedo images: Careaga and Bell

from scipy import stats

folder = "albedo_Bell"
files = glob.glob(folder+"/*.png")
files.sort()
print (files)

coef_S = 1.3
coef_H = 1



df = pd.DataFrame() 
idx = 0

for f in files:
    
    print (f)
    index = 1
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print (f)
        
    size = img.shape
    img_hsv = np.zeros((size[0], size[1], 3))
        
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            maxi = max(RGB)
            mini = min(RGB)
            
            if maxi == mini:
                mini = maxi-1
            
            if maxi == RGB[0]:
                H = 60*((G-B)/(maxi-mini))
            elif maxi == RGB[1]:
                H = 120+60*((B-R)/(maxi-mini))
            elif maxi == RGB[2]:
                H = 240+60*((R-G)/(maxi-mini)) 
                
            if maxi == 0:
                S = 0
            else:
                #s = 100*((maxi-mini)/maxi) 
                S = 100*(float((maxi-mini)/maxi))
            
            V = (maxi/255)*100  
            
            img_hsv[i,j,2] = V
            img_hsv[i,j,1] = S
            img_hsv[i,j,0] = H

    if "287" in f:
        colors = ([302, 344], [626, 305], [493, 254], [507, 253], [627, 265])

    elif "307" in f:
        colors = ([160, 653], [252, 817], [220, 712], [219, 584], [138, 659], [302, 852], [165, 710],[242, 570])

    elif "305" in f:
        colors = ([466, 276], [400, 313], [519, 326], [1013, 521], [420, 367])  

    elif "291" in f:
        colors = ([526, 708], [531, 394], [420, 607], [380, 475], [450, 707])
    
    for c in colors:
                
        x = c[0]
        y = c[1]
        
        winsz = 8   #ajustar en f(x) del soroll/textura

        color_roi = (255,0,0)
        
        #img_roi = []
        
        img = cv2.imread(f)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)

        h_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        #h_circ_np = (h_roi_np)*2
        
        h_roi = h_roi_np.flatten()
        h_mean = np.mean(h_roi)
        h_std = np.std(h_roi)
        #print (h_roi)

        h_min = np.min(h_roi)
        h_max = np.max(h_roi)
        print (h_min,   h_max)

        if h_min<0 and h_max>200:
            for i in h_roi:
                if i<0:
                    #index = h_roi.index(i)
                    #print (index)
                    j = 360 + i
                    print (j)
                    print (h_roi[index])
                    h_roi[index] = int(j)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((h_roi > (h_mean+k*h_std)) | (h_roi < (h_mean-k*h_std)))[0]
        print ("H outliers:",  outlier_indices)
        h_no_out = np.delete(h_roi, outlier_indices)
        
        h = np.mean(h_no_out)
        h_min = np.min(h_no_out)
        h_max = np.max(h_no_out)
        h_noise = h_max-h_min
        #std_h = stats.circstats.circstd(h_circ, method='angular')
        print ("H: ", h)
        

        s_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
       #Descartar outliers amb mètode Chevyshev
        s_roi = s_roi_np.flatten()
        s_mean = np.mean(s_roi)
        s_std = np.std(s_roi)
        #print (h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((s_roi > (s_mean+k*s_std)) | (s_roi < (s_mean-k*s_std)))[0]
        print ("S outliers:",  outlier_indices)
        s_no_out = np.delete(s_roi, outlier_indices)

        s = np.mean(s_no_out)
        std_s = np.std(s_no_out)
        min_s = np.min(s_no_out)
        max_s = np.max(s_no_out)
        s_noise = max_s-min_s 
        print ("S: ", s)


        v_roi = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        v = np.mean(v_roi)
        v_max = np.max(v_roi)
        std_v = np.std(v_roi)
        print ("V: ", v)
        

        rang_H = -0.08*s + 6.88  #Són les equacions de la 90D
        rang_S = 0.12*s + 0.56 
        
        print ("Rang H eq: ", rang_H)
        print ("H noise: ", h_noise)
        print ("Rang S eq: ", rang_S)
        print ("S noise: ", s_noise)

    
        c1 = (h_min, min_s, 0) #per sota només resto 1/3 part del rang_S (perquè no estic mesurant a la zona més lluminosa) i el noise perquè rang S és assimètric
        c2 = (h_max, max_s, v_max)
    
        print(c1,c2)

        mask = cv2.inRange(img_hsv, c1, c2)
        result = cv2.bitwise_and(img, img, mask=mask)
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        result_name = "../results/"+f[-7:-4]+"_Bell_"+str(winsz)+"_"+str(index)+".png"
        cv2.imwrite (result_name, result)


        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_RGB2BGR)
        img_roi_name = "../results/"+f[-7:-4]+"ROI"+str(index)+".png"
        cv2.imwrite (img_roi_name, img_roi_rgb)

        df.at[idx, "Image"] = f
        df.at[idx, "color"] = str(index)
        df.at[idx, "H"] = h
        df.at[idx, "S"] = s
        df.at[idx, "V"] = v
        df.at[idx, "H noise"] = h_noise
        df.at[idx, "S noise"] = s_noise
        df.at[idx, "coef H"] = coef_H
        df.at[idx, "H range eq"] = rang_H
        df.at[idx, "H range apl"] = (rang_H+h_noise)*coef_H
        df.at[idx, "Coef S"] = coef_S
        df.at[idx, "S range eq"] = rang_S
        df.at[idx, "S range apl"] = (rang_S+s_noise/2)*coef_S+rang_S/3+s_noise/2

        idx += 1
        
        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(img_roi)
        plt.subplot(1,2,2)
        plt.imshow(result_bgr)
        plt.show()
        
    
        index += 1

print(df)
df.to_excel("../results/Bell_results.xlsx") 

In [ ]:
#Salvador - color invariants (c1, c2, c3)

from scipy import stats

folder = "images_MP"
files = glob.glob(folder+"/*.png")
files.sort()
print (files)

coef_S = 1.3
coef_H = 1


df = pd.DataFrame() 
idx = 0

for f in files:
    
    index = 1
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print (f)
        
    size = img.shape
    img_invar = np.zeros((size[0], size[1], 3))
        
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            c1 = np.arctan(R/max(G,B))
            c2 = np.arctan(G/max(R,B))
            c3 = np.arctan(B/max(R,G))
            
            img_invar[i,j,2] = c1
            img_invar[i,j,1] = c2
            img_invar[i,j,0] = c3

    if "287" in f:
        colors = ([302, 344], [626, 305], [493, 254], [507, 253], [627, 265])

    elif "307" in f:
        colors = ([160, 653], [252, 817], [220, 712], [219, 584], [138, 659], [302, 852], [165, 710],[242, 570])

    elif "305" in f:
        colors = ([466, 276], [400, 313], [519, 326], [1013, 521], [420, 367])  

    elif "291" in f:
        colors = ([526, 708], [531, 394], [420, 607], [380, 475], [450, 707])
    
    for c in colors:
                
        x = c[0]
        y = c[1]
        
        winsz = 8   #ajustar en f(x) del soroll/textura

        color_roi = (255,0,0)
        
        #img_roi = []
        
        img = cv2.imread(f)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)

        c1_roi_np = img_invar[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        #h_circ_np = (h_roi_np)*2
        
        c1_roi = c1_roi_np.flatten()
        c1_mean = np.mean(c1_roi)
        c1_std = np.std(c1_roi)
        #print (h_roi)

        min_c1 = np.min(c1_roi)
        max_c1 = np.max(c1_roi)
        print (min_c1,   max_c1)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((c1_roi > (c1_mean+k*c1_std)) | (c1_roi < (c1_mean-k*h_std)))[0]
        print ("c1 outliers:",  outlier_indices)
        c1_no_out = np.delete(c1_roi, outlier_indices)
        
        c1 = np.mean(c1_no_out)
        min_c1 = np.min(c1_no_out)
        max_c1 = np.max(c1_no_out)
        c1_noise = max_c1-min_c1
        #std_h = stats.circstats.circstd(h_circ, method='angular')
        print ("c1: ", c1)
        

        c2_roi_np = img_invar[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
       #Descartar outliers amb mètode Chevyshev
        c2_roi = c2_roi_np.flatten()
        c2_mean = np.mean(c2_roi)
        c2_std = np.std(c2_roi)
        #print (h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((c2_roi > (c2_mean+k*c2_std)) | (c2_roi < (c2_mean-k*c2_std)))[0]
        print ("c2 outliers:",  outlier_indices)
        c2_no_out = np.delete(c2_roi, outlier_indices)

        c2 = np.mean(c2_no_out)
        std_c2 = np.std(c2_no_out)
        min_c2 = np.min(c2_no_out)
        max_c2 = np.max(c2_no_out)
        c2_noise = max_c2-min_c2 
        print ("c2: ", c2)


        c3_roi_np = img_invar[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        
       #Descartar outliers amb mètode Chevyshev
        c3_roi = c3_roi_np.flatten()
        c3_mean = np.mean(c3_roi)
        c3_std = np.std(c3_roi)
        #print (h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((c3_roi > (c3_mean+k*c3_std)) | (c3_roi < (c3_mean-k*c3_std)))[0]
        print ("c3 outliers:",  outlier_indices)
        c3_no_out = np.delete(c3_roi, outlier_indices)

        c3 = np.mean(c3_no_out)
        std_c3 = np.std(c3_no_out)
        min_c3 = np.min(c3_no_out)
        max_c3 = np.max(c3_no_out)
        c3_noise = max_c3-min_c3 
        print ("c3: ", c3)

    
        limit1 = (min_c1, min_c2, min_c3) 
        limit2 = (max_c1, max_c2, max_c3)
    
        print(limit1,limit2)

        mask = cv2.inRange(img_invar, limit1, limit2)
        result = cv2.bitwise_and(img, img, mask=mask)
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        result_name = "../results/"+f[-13:-10]+"_Salvador_"+str(winsz)+"_"+str(index)+".png"
        cv2.imwrite (result_name, result)


        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_RGB2BGR)
        img_roi_name = "../results/"+f[-13:-10]+"ROI"+str(index)+".png"
        cv2.imwrite (img_roi_name, img_roi_rgb)

        df.at[idx, "Image"] = f
        df.at[idx, "color"] = str(index)
        df.at[idx, "c1"] = c1
        df.at[idx, "c2"] = c2
        df.at[idx, "c3"] = c3
        df.at[idx, "c1 noise"] = c1_noise
        df.at[idx, "c2 noise"] = c2_noise
        df.at[idx, "c3 noise"] = c3_noise


        idx += 1
        
        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(img_roi)
        plt.subplot(1,2,2)
        plt.imshow(result_bgr)
        plt.show()
        
    
        index += 1

print(df)
df.to_excel("../results/Salvador_results.xlsx") 